## Sistema de Temporadas

Utilizando el Dataframe con los datos de todos los partidos, creamos el sistema de temporadas en archivos individuales

In [1]:
import pandas as pd
import os
dfpartits=pd.read_pickle('dfPartits.pkl')
dfequips=pd.read_csv('team_sp.csv')

#### Cambiar las id a los nombres de los equipos

In [2]:
lequips=dfequips['team_long_name'].to_list()
lid=dfequips['team_api_id'].to_list()
lidequips=list(zip(lequips,lid)) 

Funciones para poder aplicar el cambio

In [3]:
def canviar_noms_home(fila):
    x=''
    id_home=fila['home_team_api_id']
    for i in lidequips:
        if id_home==int(i[1]):
            x=i[0]
    return x

def canviar_noms_away(fila):
    x=''
    id_away=fila['away_team_api_id']
    for i in lidequips:
        if id_away==int(i[1]):
            x=i[0]
    return x

Aplicar las funciones

In [4]:
dfpartits['home_team_api_id'] = dfpartits.apply(canviar_noms_home, axis=1)
dfpartits['away_team_api_id'] = dfpartits.apply(canviar_noms_away, axis=1)

##### Partidos ganados y perdidos

In [5]:
def game_counter(fila):
    goal_home, goal_away = fila['home_team_goal'], fila['away_team_goal']
    win_home=0
    lose_home=0
    win_away=0
    lose_away=0
    drawn_home=0
    drawn_away=0
    played_home=1
    played_away=1
    if goal_home>goal_away:
        win_home=1
        lose_away=1
    elif goal_home<goal_away:
        win_away=1
        lose_home=1
    else:
        drawn_home=1
        drawn_away=1
        
    return pd.Series([win_home, lose_home, win_away, lose_away, drawn_home, drawn_away, played_home, played_away])

Aplicar la función

In [6]:
dfpartits[['win_home', 'lose_home', 'win_away', 'lose_away' , 'drawn_home', 'drawn_away','played_home','played_away']] = dfpartits.apply(game_counter, axis=1)

##### Diferencia de goles

In [7]:
dfpartits['home_goal_difference']=dfpartits['home_team_goal']-dfpartits['away_team_goal']
dfpartits['away_goal_difference']=dfpartits['away_team_goal']-dfpartits['home_team_goal']

##### Tiros totales

In [8]:
dfpartits['home_total_shots']=dfpartits['shoton_home']+dfpartits['shotoff_home']
dfpartits['away_total_shots']=dfpartits['shoton_away']+dfpartits['shotoff_away']

##### Puntuar a partir de la victoria o derrota

Funciones para poder aplicar el cambio

In [9]:
def points_counter(fila):
    goal_home, goal_away = fila['home_team_goal'], fila['away_team_goal']
    points_home=0
    points_away=0
    if goal_home>goal_away:
        points_home=3
    elif goal_home<goal_away:
        points_away=3
    else:
        points_home=1
        points_away=1
        
    return pd.Series([points_home, points_away])

Aplicar la función

In [10]:
dfpartits[['points_home', 'points_away']] = dfpartits.apply(points_counter, axis=1)

#### Formación de cada temporada separada por Dataframes individuales

Utilizamos un "for" para hacer y guardar los Dataframes de las diferentes temporadas en archivos induviduales 

In [11]:
temp=['2008/2009','2009/2010','2010/2011','2011/2012','2012/2013','2013/2014','2014/2015','2015/2016']
tempstr=['2008-2009','2009-2010','2010-2011','2011-2012','2012-2013','2013-2014','2014-2015','2015-2016']
for i in range(len(temp)):
    temporada=temp[i]
    dftemp=dfpartits[dfpartits['season']==temporada]
    home_stats = dftemp[['home_team_api_id','played_home','win_home','drawn_home','lose_home', 'home_goal_difference','home_team_goal','points_home','foulcommit_home',"red_card_home", "yellow_card_home", "corner_home",'shoton_home','shotoff_home','home_total_shots']].rename(columns={'home_team_api_id': 'Club' ,'played_home': 'Partidos', 'win_home': 'V', 'drawn_home': 'E', 'lose_home': 'D', 'home_goal_difference': 'DG', 'home_team_goal': 'Goles', 'points_home': 'Pts', 'foulcommit_home': 'Faltas',"red_card_home": 'T.R', "yellow_card_home": 'T.A', "corner_home": 'Corner','shoton_home': 'TaP','shotoff_home': 'Tf','home_total_shots': 'Tt'})
    away_stats = dftemp[['away_team_api_id','played_away','win_away','drawn_away','lose_away', 'away_goal_difference','away_team_goal','points_away','foulcommit_away',"red_card_away", "yellow_card_away", "corner_away",'shoton_away','shotoff_away','away_total_shots']].rename(columns={'away_team_api_id': 'Club', 'played_away': 'Partidos', 'win_away': 'V', 'drawn_away': 'E', 'lose_away': 'D', 'away_goal_difference': 'DG', 'away_team_goal': 'Goles', 'points_away': 'Pts', 'foulcommit_away': 'Faltas',"red_card_away": 'T.R', "yellow_card_away": 'T.A', "corner_away": 'Corner','shoton_away': 'TaP','shotoff_away': 'Tf','away_total_shots': 'Tt'})
    stats_totals = pd.concat([home_stats, away_stats])
    stats_totals.fillna(0, inplace=True)
    
    Lliga = stats_totals.groupby('Club', dropna=False).sum().reset_index()
    Temp=Lliga.sort_values(['Pts','V','DG'],ascending=False) 
    
    #Eliminar # si se hace con Linux/Ubuntu
    #ruta_pickle = "/home/marc.mogollon/data/Projecte 1/temp"+tempstr[i]+".pkl"
    #os.makedirs("/home/marc.mogollon/data/Projecte 1/temp"+tempstr[i]+".pkl", exist_ok=True)
    
    #Eliminar # si se hace con Windows
    ruta_pickle = "/Users/marcm/Projecte 1/Temp"+tempstr[i]+".pkl"
    
    Temp.to_pickle(ruta_pickle)
    dfpartits.to_pickle("/Users/marcm/Projecte 1/dfmodelpred.pkl")

C:\Users\marcm\AppData\Local\Temp\ipykernel_6156\732488665.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stats_totals.fillna(0, inplace=True)
C:\Users\marcm\AppData\Local\Temp\ipykernel_6156\732488665.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stats_totals.fillna(0, inplace=True)
C:\Users\marcm\AppData\Local\Temp\ipykernel_6156\732488665.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future beh

Visualización de las temporadas

In [12]:
dfaou=pd.read_csv('dfmodelpred.csv')

In [13]:
dftemp

,season,stage,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,shoton_home,shoton_away,shotoff_home,...,drawn_home,drawn_away,played_home,played_away,home_goal_difference,away_goal_difference,home_total_shots,away_total_shots,points_home,points_away
2660,2015/2016,1,2015-08-23 00:00:00,Levante UD,RC Celta de Vigo,1,2,3,7,6,...,0,0,1,1,-1,1,9,13,0,3
2661,2015/2016,1,2015-08-22 00:00:00,Atlético Madrid,UD Las Palmas,1,0,5,1,6,...,0,0,1,1,1,-1,11,7,3,0
2662,2015/2016,1,2015-08-21 00:00:00,Málaga CF,Sevilla FC,0,0,9,3,13,...,1,1,1,1,0,0,22,9,1,1
2663,2015/2016,1,2015-08-23 00:00:00,Athletic Club de Bilbao,FC Barcelona,0,1,4,5,4,...,0,0,1,1,-1,1,8,9,0,3
2664,2015/2016,1,2015-08-24 00:00:00,Granada CF,SD Eibar,1,3,6,4,5,...,0,0,1,1,-2,2,11,11,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3035,2015/2016,9,2015-10-25 00:00:00,Atlético Madrid,Valencia CF,2,1,9,2,5,...,0,0,1,1,1,-1,14,6,3,0
3036,2015/2016,9,2015-10-24 00:00:00,Málaga CF,RC Deportivo de La Coruña,2,0,3,4,3,...,0,0,1,1,2,-2,6,9,3,0
3037,2015/2016,9,2015-10-26 00:00:00,Athletic Club de Bilbao,Real Sporting de Gijón,3,0,3,2,12,...,0,0,1,1,3,-3,15,6,3,0
3038,2015/2016,9,2015-10-24 00:00:00,Granada CF,Real Betis Balompié,1,1,7,3,6,...,1,1,1,1,0,0,13,3,1,1


In [14]:
l=['Temp2008-2009.pkl','Temp2009-2010.pkl','Temp2010-2011.pkl','Temp2011-2012.pkl','Temp2012-2013.pkl','Temp2013-2014.pkl','Temp2014-2015.pkl','Temp2015-2016.pkl']
df1=pd.read_pickle(l[0])
df2=pd.read_pickle(l[2])
df3=pd.read_pickle(l[2])
df4=pd.read_pickle(l[3])
df5=pd.read_pickle(l[4])
df6=pd.read_pickle(l[5])
df7=pd.read_pickle(l[6])
df8=pd.read_pickle(l[7])
df1

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
4,FC Barcelona,38,27,6,5,70,105,87,282,3,73,241,288,282,570
13,Real Madrid CF,38,25,3,10,31,83,78,243,6,114,181,266,243,509
16,Sevilla FC,38,21,7,10,15,54,70,217,4,101,158,158,217,375
1,Atlético Madrid,38,20,7,11,23,80,67,154,1,108,128,142,154,296
19,Villarreal CF,38,18,11,9,7,61,65,127,1,97,123,129,127,256
18,Valencia CF,38,18,8,12,14,68,62,149,1,117,139,167,149,315
7,RC Deportivo de La Coruña,38,16,10,12,1,48,58,70,3,79,83,80,70,150
6,Málaga CF,38,15,10,13,-4,55,55,67,6,88,58,57,67,124
10,RCD Mallorca,38,14,9,15,-7,53,51,59,3,103,57,56,59,115
9,RCD Espanyol,38,12,11,15,-3,46,47,68,2,101,51,69,68,137


In [15]:
df2

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
3,FC Barcelona,38,30,6,2,74,95,96,228,1,72,256,250,228,478
12,Real Madrid CF,38,29,5,4,69,102,92,289,3,98,259,307,289,596
18,Valencia CF,38,21,8,9,20,64,71,179,3,116,222,179,179,358
19,Villarreal CF,38,18,8,12,10,54,62,185,2,92,142,186,185,371
0,Athletic Club de Bilbao,38,18,4,16,4,59,58,181,3,105,190,150,181,331
1,Atlético Madrid,38,17,7,14,9,62,58,219,5,107,227,189,219,408
16,Sevilla FC,38,17,7,14,1,62,58,196,3,105,173,182,196,378
9,RCD Espanyol,38,15,4,19,-9,46,49,85,3,107,89,92,85,177
2,CA Osasuna,38,13,8,17,-1,45,47,117,2,113,120,86,117,203
14,Real Sporting de Gijón,38,11,14,13,-7,35,47,69,3,99,52,49,69,118


In [16]:
df3

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
3,FC Barcelona,38,30,6,2,74,95,96,228,1,72,256,250,228,478
12,Real Madrid CF,38,29,5,4,69,102,92,289,3,98,259,307,289,596
18,Valencia CF,38,21,8,9,20,64,71,179,3,116,222,179,179,358
19,Villarreal CF,38,18,8,12,10,54,62,185,2,92,142,186,185,371
0,Athletic Club de Bilbao,38,18,4,16,4,59,58,181,3,105,190,150,181,331
1,Atlético Madrid,38,17,7,14,9,62,58,219,5,107,227,189,219,408
16,Sevilla FC,38,17,7,14,1,62,58,196,3,105,173,182,196,378
9,RCD Espanyol,38,15,4,19,-9,46,49,85,3,107,89,92,85,177
2,CA Osasuna,38,13,8,17,-1,45,47,117,2,113,120,86,117,203
14,Real Sporting de Gijón,38,11,14,13,-7,35,47,69,3,99,52,49,69,118


In [17]:
df4

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
13,Real Madrid CF,38,32,4,2,89,121,100,299,1,93,227,272,299,571
3,FC Barcelona,38,28,7,3,85,114,91,221,1,79,263,289,221,510
18,Valencia CF,38,17,10,11,15,59,61,208,5,125,206,165,208,373
7,Málaga CF,38,17,7,14,1,54,58,99,4,74,112,123,99,222
1,Atlético Madrid,38,15,11,12,7,53,56,198,2,127,195,191,198,389
6,Levante UD,38,16,7,15,4,54,55,75,1,115,79,62,75,137
2,CA Osasuna,38,13,15,10,-17,44,54,88,2,103,82,85,88,173
9,RCD Mallorca,38,14,10,14,-4,42,52,84,4,118,91,64,84,148
17,Sevilla FC,38,13,11,14,1,48,50,223,3,111,184,197,223,420
0,Athletic Club de Bilbao,38,12,13,13,-3,49,49,110,2,109,155,100,110,210


In [18]:
df5

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
3,FC Barcelona,38,32,4,2,75,115,100,0,1,58,0,0,0,0
14,Real Madrid CF,38,26,7,5,61,103,85,0,2,92,0,0,0,0
1,Atlético Madrid,38,23,7,8,34,65,76,0,0,101,0,0,0,0
15,Real Sociedad,38,18,12,8,21,70,66,0,0,93,0,0,0,0
19,Valencia CF,38,19,8,11,13,67,65,0,4,121,0,0,0,0
7,Málaga CF,38,16,9,13,3,53,57,0,4,94,0,0,0,0
13,Real Betis Balompié,38,16,8,14,1,57,56,0,3,118,0,0,0,0
12,Rayo Vallecano,38,16,5,17,-16,50,53,0,2,139,0,0,0,0
18,Sevilla FC,38,14,8,16,4,58,50,0,6,97,0,0,0,0
4,Getafe CF,38,13,8,17,-14,43,47,0,6,114,0,0,0,0


In [19]:
df6

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
1,Atlético Madrid,38,28,6,4,51,77,90,0,2,96,0,0,0,0
4,FC Barcelona,38,27,6,5,67,100,87,8,0,71,9,19,8,27
13,Real Madrid CF,38,27,6,5,66,104,87,9,2,75,4,11,9,20
0,Athletic Club de Bilbao,38,20,10,8,27,66,70,0,1,79,0,0,0,0
16,Sevilla FC,38,18,9,11,17,69,63,0,3,116,0,0,0,0
19,Villarreal CF,38,17,8,13,16,60,59,0,1,74,0,0,0,0
14,Real Sociedad,38,16,11,11,7,62,59,0,1,76,0,0,0,0
9,RC Celta de Vigo,38,14,7,17,-5,49,49,0,1,84,0,0,0,0
18,Valencia CF,38,13,10,15,-2,51,49,0,0,97,0,0,0,0
7,Levante UD,38,12,12,14,-8,35,48,0,7,110,0,0,0,0


In [20]:
df7

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
4,FC Barcelona,38,30,4,4,89,110,94,255,2,68,236,272,255,527
13,Real Madrid CF,38,30,2,6,80,118,92,288,2,86,246,281,288,569
1,Atlético Madrid,38,23,9,6,38,67,78,178,1,114,195,194,178,372
18,Valencia CF,38,22,11,5,38,70,77,203,4,108,168,176,203,379
16,Sevilla FC,38,23,7,8,26,71,76,183,1,117,200,219,183,402
19,Villarreal CF,38,16,12,10,11,48,60,223,1,95,227,234,223,457
0,Athletic Club de Bilbao,38,15,10,13,1,42,55,196,1,99,208,211,196,407
9,RC Celta de Vigo,38,13,12,13,3,47,51,220,4,117,240,223,220,443
8,Málaga CF,38,14,8,16,-6,42,50,205,5,109,200,211,205,416
12,Rayo Vallecano,38,15,4,19,-22,46,49,218,2,109,197,224,218,442


In [21]:
df8

,Club,Partidos,V,E,D,DG,Goles,Pts,Faltas,T.R,T.A,Corner,TaP,Tf,Tt
2,FC Barcelona,38,29,4,5,83,112,91,255,1,66,221,254,255,509
12,Real Madrid CF,38,28,6,4,76,110,90,297,2,71,257,283,297,580
1,Atlético Madrid,38,28,4,6,45,63,88,224,1,90,208,197,224,421
19,Villarreal CF,38,18,10,10,9,44,64,153,1,99,175,145,153,298
0,Athletic Club de Bilbao,38,18,8,12,13,58,62,202,4,85,208,181,202,383
7,RC Celta de Vigo,38,17,9,12,-8,51,60,187,3,113,204,201,187,388
16,Sevilla FC,38,14,10,14,1,51,52,199,2,101,252,212,199,411
13,Real Sociedad,38,13,9,16,-3,45,48,196,2,106,199,196,196,392
6,Málaga CF,38,12,12,14,3,38,48,194,1,108,193,206,194,400
11,Real Betis Balompié,38,11,12,15,-18,34,45,168,0,110,164,167,168,335


In [22]:
dfpartits

,season,stage,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,shoton_home,shoton_away,shotoff_home,...,drawn_home,drawn_away,played_home,played_away,home_goal_difference,away_goal_difference,home_total_shots,away_total_shots,points_home,points_away
0,2008/2009,1,2008-08-30 00:00:00,Valencia CF,RCD Mallorca,3,0,5,4,8,...,0,0,1,1,3,-3,13,14,3,0
1,2008/2009,1,2008-08-31 00:00:00,CA Osasuna,Villarreal CF,1,1,<NA>,<NA>,<NA>,...,1,1,1,1,0,0,<NA>,<NA>,1,1
2,2008/2009,1,2008-08-31 00:00:00,RC Deportivo de La Coruña,Real Madrid CF,2,1,2,12,5,...,0,0,1,1,1,-1,7,17,3,0
3,2008/2009,1,2008-08-31 00:00:00,CD Numancia,FC Barcelona,1,0,2,6,3,...,0,0,1,1,1,-1,5,21,3,0
4,2008/2009,1,2008-08-31 00:00:00,Racing Santander,Sevilla FC,1,1,<NA>,<NA>,<NA>,...,1,1,1,1,0,0,<NA>,<NA>,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3035,2015/2016,9,2015-10-25 00:00:00,Atlético Madrid,Valencia CF,2,1,9,2,5,...,0,0,1,1,1,-1,14,6,3,0
3036,2015/2016,9,2015-10-24 00:00:00,Málaga CF,RC Deportivo de La Coruña,2,0,3,4,3,...,0,0,1,1,2,-2,6,9,3,0
3037,2015/2016,9,2015-10-26 00:00:00,Athletic Club de Bilbao,Real Sporting de Gijón,3,0,3,2,12,...,0,0,1,1,3,-3,15,6,3,0
3038,2015/2016,9,2015-10-24 00:00:00,Granada CF,Real Betis Balompié,1,1,7,3,6,...,1,1,1,1,0,0,13,3,1,1
